In [1]:
!pip install transformers datasets -q

DEPRECATION: python-apt 2.4.0-elementary9-ubuntu7.1 has a non-standard version number. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of python-apt or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063


# PARAMETERS

In [2]:
TOKENIZER_NAME = "gpt2"
CONTEXT_SIZE = 2048

# Heavy Stuff Loading
Here, code that loads the dataset, or other intensive parts

In [3]:
from datasets import load_dataset
load_dataset("Open-Orca/OpenOrca")


/home/rohethan/.local/lib/python3.10/site-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by mode='default'.
  table = cls._concat_blocks(blocks, axis=0)


DatasetDict({
    train: Dataset({
        features: ['id', 'system_prompt', 'question', 'response'],
        num_rows: 4233923
    })
})

# The model
Here the code to define the model builder functions.

In [4]:
import tensorflow as tf
from tensorflow.keras import layers as L

def build_llm_model(context_win, vocab_size, embedding_size , n_attention_heads, dense_units, dropout_rate=0.15, normalisation_epsilon=1e-3):
    '''

    :param context_win: context window size
    :param vocab_size: vocab size
    :param embedding_size: embedding size
    :param attention_heads: number of attention heads
    :return: a compiled tensorflow model.

    This function builds the model.
    This model has 4 attention blocks.
    '''
    # =-=-=-=-=-=-=-=- INPUTS LAYERS
    context_input_tokens = tf.keras.Input(shape=(context_win,), dtype=tf.int64)
    speaker_input_tokens = tf.keras.Input(shape=(context_win,), dtype=tf.int64)

    embedded_context = L.Embedding(vocab_size, embedding_size)(context_input_tokens)
    embedded_speaker = L.Embedding(4, embedding_size)(speaker_input_tokens)
    embeds = embedded_context + embedded_speaker

    # =-=-=-=-=-=-=-=- FIRST ATTENTION BLOCK

    attention_block_1 = L.MultiHeadAttention(num_heads=n_attention_heads, key_dim=embedding_size)(embeds, embeds)
    attention_block_1 = L.LayerNormalization(epsilon=normalisation_epsilon)(L.Dropout(dropout_rate)(attention_block_1))
    dense_layer_block_1 = L.Dense(units=dense_units, activation='sigmoid')(attention_block_1)
    dense_output_block_1 = L.Dense(units=embedding_size)(dense_layer_block_1)

    dense_output_block_1 = L.Dropout(dropout_rate)(dense_output_block_1)
    embeds = L.LayerNormalization(epsilon=normalisation_epsilon)(embeds + dense_output_block_1)

    # =-=-=-=-=-=-=-=- SECOND ATTENTION BLOCK
    attention_block_2 = L.MultiHeadAttention(num_heads=n_attention_heads, key_dim=embedding_size)(embeds, embeds)
    attention_block_2 = L.LayerNormalization(epsilon=normalisation_epsilon)(L.Dropout(dropout_rate)(attention_block_2))
    dense_layer_block_2 = L.Dense(units=dense_units, activation='sigmoid')(attention_block_2)
    dense_output_block_2 = L.Dense(units=embedding_size)(dense_layer_block_2)

    dense_output_block_2 = L.Dropout(dropout_rate)(dense_output_block_2)
    embeds = L.LayerNormalization(epsilon=normalisation_epsilon)(embeds + dense_output_block_2)

    # =-=-=-=-=-=-=-=- THIRD ATTENTION BLOCK
    attention_block_3 = L.MultiHeadAttention(num_heads=n_attention_heads, key_dim=embedding_size)(embeds, embeds)
    attention_block_3 = L.LayerNormalization(epsilon=normalisation_epsilon)(L.Dropout(dropout_rate)(attention_block_3))
    dense_layer_block_3 = L.Dense(units=dense_units, activation='sigmoid')(attention_block_3)
    dense_output_block_3 = L.Dense(units=embedding_size)(dense_layer_block_3)

    dense_output_block_3 = L.Dropout(dropout_rate)(dense_output_block_3)
    embeds = L.LayerNormalization(epsilon=normalisation_epsilon)(embeds + dense_output_block_3)


    # =-=-=-=-=-=-=-=- FOURTH ATTENTION BLOCK
    attention_block_4 = L.MultiHeadAttention(num_heads=n_attention_heads, key_dim=embedding_size)(embeds, embeds)
    attention_block_4 = L.LayerNormalization(epsilon=normalisation_epsilon)(L.Dropout(dropout_rate)(attention_block_4))
    dense_layer_block_4 = L.Dense(units=dense_units, activation='sigmoid')(attention_block_4)
    dense_output_block_4 = L.Dense(units=embedding_size)(dense_layer_block_4)

    dense_output_block_4 = L.Dropout(dropout_rate)(dense_output_block_4)
    embeds = L.LayerNormalization(epsilon=normalisation_epsilon)(embeds + dense_output_block_4)


    # =-=-=-=-=-=-=-=- OUTPUT LAYERS
    pooled_output = L.GlobalAveragePooling1D()(embeds)
    final_output = L.Dense(units=vocab_size, activation='softmax')(pooled_output)

    model = tf.keras.models.Model(inputs=[context_input_tokens, speaker_input_tokens], outputs=final_output)

    # Compile the model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=["accuracy"])

    return model

2024-06-09 13:08:17.969226: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-06-09 13:08:17.990769: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-09 13:08:18.104366: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-09 13:08:18.104397: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-09 13:08:18.105051: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

# The Dataset
Here code to manage the dataset generator and generator to tfds transform

In [5]:
import tiktoken
# Dataset generator and processing
# Assign speaker IDs
SYSTEM_ID = 1
USER_ID = 2
MODEL_ID = 3
PADDING_ID = 0

tokenizer = tiktoken.get_encoding("gpt2")
END_OF_TEXT_TOKEN = tokenizer.eot_token

def lenreg(list, size=1024, padding_idx=50257):
    return (list + [padding_idx for _ in range(size)])[:size]

def create_speaker_context(seq_length, speaker_id, context_size, padding_idx=50257):
    return [speaker_id if i < seq_length and i != padding_idx else padding_idx for i in range(context_size)]

def generator(context_size=CONTEXT_SIZE):  # Default context window size to 2048
    print("[GENERATOR] Loading dataset and tokenizer")
    dataset = load_dataset("Open-Orca/OpenOrca")
    print("[GENERATOR] dataset loaded")
    
    print("[GENERATOR] tokenizer loaded")

    for features in dataset["train"]:
        sp = tokenizer.encode(features["system_prompt"])
        sp.append(END_OF_TEXT_TOKEN)
        up = tokenizer.encode(features["question"])
        up.append(END_OF_TEXT_TOKEN)
        ma = tokenizer.encode(features["response"])
        ma.append(END_OF_TEXT_TOKEN)
        #print(features["system_prompt"], features["question"], features["response"])
        token_context = sp + up
        speaker_context = [SYSTEM_ID]*len(sp) + [USER_ID]*len(up)
        tensorize = tf.convert_to_tensor
        for ma_token in ma:
            regulated_token_context = tensorize(lenreg(token_context, size=context_size), dtype=tf.int64)
            regulated_speaker_context = tensorize(lenreg(speaker_context, size=context_size, padding_idx=0), dtype=tf.int64)
            yield (regulated_token_context, regulated_speaker_context), ma_token
            token_context.append(ma_token)
            speaker_context.append(MODEL_ID)


output_signature = (
    (
        tf.TensorSpec(shape=(CONTEXT_SIZE), dtype=tf.int64),
        tf.TensorSpec(shape=(CONTEXT_SIZE), dtype=tf.int64)
    ),
    tf.TensorSpec(shape=(), dtype=tf.int64)
)
dataset = tf.data.Dataset.from_generator(generator, output_signature=output_signature, args=(CONTEXT_SIZE,))


# The Training
Cells related to prepping to train and train

In [8]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

model = build_llm_model(CONTEXT_SIZE,
                        tokenizer.n_vocab + 1,
                        1536,
                        2,
                        3072)

model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 2048)]               0         []                            
                                                                                                  
 input_4 (InputLayer)        [(None, 2048)]               0         []                            
                                                                                                  
 embedding_2 (Embedding)     (None, 2048, 1536)           7719628   ['input_3[0][0]']             
                                                          8                                       
                                                                                                  
 embedding_3 (Embedding)     (None, 2048, 1536)           6144      ['input_4[0][0]']       

In [9]:
from tensorflow.keras.callbacks import *
# TensorBoard callback
tensorboard_callback = TensorBoard(log_dir="./logs", histogram_freq=1)

# Terminate training on nan. (Don't spend additional compute ressources)
nan_stop = TerminateOnNaN()

# ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath="./checkpoints/" + "ckpt_{epoch:03d}-{loss:.3f}.ckpt",
    save_weights_only=True,
    save_freq=4096,# Save the model every 2 epochs
    verbose=1)

dataset = dataset.batch(2).prefetch(2)
with tf.device('/CPU:0'):
    model.fit(dataset, epochs=100, steps_per_epoch=2048, verbose=1, batch_size=2, callbacks=[tensorboard_callback, nan_stop, checkpoint_callback])
    model.save("test")

Epoch 1/100


ValueError: in user code:

    File "/home/rohethan/.local/lib/python3.10/site-packages/keras/src/engine/training.py", line 1377, in train_function  *
        return step_function(self, iterator)
    File "/home/rohethan/.local/lib/python3.10/site-packages/keras/src/engine/training.py", line 1360, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/rohethan/.local/lib/python3.10/site-packages/keras/src/engine/training.py", line 1349, in run_step  **
        outputs = model.train_step(data)
    File "/home/rohethan/.local/lib/python3.10/site-packages/keras/src/engine/training.py", line 1126, in train_step
        y_pred = self(x, training=True)
    File "/home/rohethan/.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None

    ValueError: Exception encountered when calling layer 'query' (type EinsumDense).
    
    Shape must be rank